# Evaluating the Final Model on 2025-03-01 to 2026-02-28

We had initially held out 2025-03-01 to 2026-02-28. We shall use TimeSeriesSplit to see how our chosen model performs. We have chosen Prophet for Manhattan since the other models did not provide any significant improvements while also increasing modeling complexity greatly. For example, we see that the hybrid XGBoost and Prophet model had improved the RMSE by roughly 0.2.

## Import and Load Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import xgboost as xgb
from xgboost import plot_importance

In [5]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2025-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to MANHATTAN

rs = rs[rs['borough']=='MANHATTAN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])


In [6]:
rs['created_date'] = pd.to_datetime(rs['created_date'])
start = rs['created_date'].min()
end = rs['created_date'].max()
rs = (rs.set_index('created_date').reindex(pd.date_range(start, end, freq='D'), fill_value=0).rename_axis('created_date').reset_index())
rs

,created_date,count
0,2020-01-01,4
1,2020-01-02,7
2,2020-01-03,16
3,2020-01-04,10
4,2020-01-05,5
...,...,...
2246,2026-02-24,7
2247,2026-02-25,8
2248,2026-02-26,9
2249,2026-02-27,17


In [8]:
# Create a date range covering 2020 to end
date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

# Build the DataFrame in the same structure as your original
federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1,
})

holidays = federal_holidays

## Evaluate the Model

In [9]:
# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    y_pred = np.round(y_pred)

    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Calculate MAPE
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# Convert results to a datafrane
prophet_results_df = pd.DataFrame(results)

15:26:44 - cmdstanpy - INFO - Chain [1] start processing
15:26:44 - cmdstanpy - INFO - Chain [1] done processing
15:26:45 - cmdstanpy - INFO - Chain [1] start processing
15:26:45 - cmdstanpy - INFO - Chain [1] done processing
15:26:46 - cmdstanpy - INFO - Chain [1] start processing
15:26:46 - cmdstanpy - INFO - Chain [1] done processing
15:26:46 - cmdstanpy - INFO - Chain [1] start processing
15:26:47 - cmdstanpy - INFO - Chain [1] done processing
15:26:47 - cmdstanpy - INFO - Chain [1] start processing
15:26:47 - cmdstanpy - INFO - Chain [1] done processing
15:26:48 - cmdstanpy - INFO - Chain [1] start processing
15:26:48 - cmdstanpy - INFO - Chain [1] done processing
15:26:49 - cmdstanpy - INFO - Chain [1] start processing
15:26:49 - cmdstanpy - INFO - Chain [1] done processing
15:26:49 - cmdstanpy - INFO - Chain [1] start processing
15:26:49 - cmdstanpy - INFO - Chain [1] done processing
15:26:50 - cmdstanpy - INFO - Chain [1] start processing
15:26:50 - cmdstanpy - INFO - Chain [1]

## Results

In [10]:
prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]

In [11]:
prophet_results_df

,fold,rmse,mape
0,0,4.750940,3.597222e-01
1,1,7.372730,3.214702e-01
2,2,3.770184,2.534078e-01
3,3,3.946065,2.838269e-01
4,4,4.472136,1.851062e-01
5,5,6.256425,4.008408e-01
6,6,5.028490,2.345638e-01
7,7,6.369571,1.960804e-01
8,8,6.017831,2.792741e-01
9,9,5.126960,3.286943e-01
